# Python 3.14t (free-threading / nogil) 工作模板

本模板面向 `devcontainer-base:conda-llvm-latest` 镜像（main 环境，Python 3.14t free-threading 构建）。

**核心逻辑已提取为可复用库** [`scripts/nogil_kit.py`](../scripts/nogil_kit.py)：环境自检 / GIL 诊断 / kernel 注册 / 性能测量均可在任意项目里 `from nogil_kit import ...` 直接调用；本 notebook 是它的交互式薄封装。

**使用流程**：
1. 依次运行下方单元 —— 环境自检 → GIL 诊断 →（可选）注册 nogil kernel
2. 若诊断显示 GIL 被拉起（如 jupyter 生态的 `_brotli` 所致），按方案 B 用进程隔离跑重计算
3. 切换到注册好的 `Python 3.14t (nogil)` kernel 重跑自检，确认全绿

> 关键事实：GIL 状态是**进程级一次性保险丝**——一旦因 import 未声明 `Py_mod_gil` 的模块被拉起，无法在当前进程内关闭。kernel 启动前的 `PYTHON_GIL=0` 只是初始请求，防不住运行时拉起。

In [ ]:
# 1. 环境自检：构建类型 / GIL 状态 / 核心数（nogil_kit.env_report）
import sys
from pathlib import Path

# 定位 scripts/nogil_kit.py（仓库根或 examples/ 目录启动均可）
KIT_DIR = next((p for p in (Path("scripts"), Path("../scripts"))
                if (p / "nogil_kit.py").exists()), None)
assert KIT_DIR, "未找到 scripts/nogil_kit.py —— 请从仓库根或 examples/ 目录启动 Jupyter"
sys.path.insert(0, str(KIT_DIR.resolve()))

from nogil_kit import env_report, format_env_report

r = env_report()
print(format_env_report(r))
assert r["free_threading"], "当前 kernel 非 free-threading 构建，请切换到 main 环境（Python 3.14t）kernel"

## 2. GIL 状态诊断（`nogil_kit.diagnose` → `check_gil_state.py`）

若 GIL 已被拉起，脚本会用「金丝雀子进程」逐个重 import 已加载的 C 扩展，**定位肇事模块**（如 `_brotli`）。

In [ ]:
# 以子进程方式调用（避免 argparse 解析到 ipykernel 的启动参数）
from nogil_kit import diagnose

rc = diagnose()   # 退出码: 0=健康 1=GIL被拉起 2=非ft构建
print(f"\n诊断退出码: {rc}")

## 3. 注册 nogil kernel（`nogil_kit.register_nogil_kernel`，幂等，可选）

注册一个 `PYTHON_GIL=0` 的 kernelspec。注册后**重启 JupyterLab**，在 Kernel 菜单选择 `Python 3.14t (nogil)`。

注意：`PYTHON_GIL=0` 只保证初始状态；若该 kernel 又 import 了未适配模块，GIL 仍会被拉起（用上一单元复诊）。

In [ ]:
from nogil_kit import register_nogil_kernel

spec_dir, created = register_nogil_kernel()
tag = "OK" if created else "i"
note = "（新建，env PYTHON_GIL=0）" if created else "（已存在）"
print(f"[{tag}] kernel 'Python 3.14t (nogil)': {spec_dir}{note}")
print("[i] 重启 JupyterLab 后在 Kernel 菜单切换")

## 4. 多线程扩展快速基准（`nogil_kit.quick_thread_scaling`）

纯 Python CPU 密集任务（素数计数），观察 nogil 下线程加速比。GIL 被拉起时此单元会退化为 ~1x——那正是诊断单元要告诉你的事。

In [ ]:
from nogil_kit import quick_thread_scaling

RANGE = 200_000
sp = quick_thread_scaling(RANGE)          # {"1": 1.0, "2": 2.3, "4": ..., "8": ...}
base = sp.pop("_base_seconds")
print(f"单线程        : {base:.3f}s  1.00x")
for w, ratio in sorted(sp.items(), key=lambda kv: int(kv[0])):
    print(f"{w:>2} workers     : {base / ratio:.3f}s  {ratio:5.2f}x")

## 5. 重计算姿势：进程隔离（方案 B，`nogil_kit.pool_compare`，推荐）

kernel 进程 GIL 状态无所谓——worker 子进程各自以干净状态启动（默认 GIL off），互不污染。对照 ThreadPool / ProcessPool 的真实开销差。

In [ ]:
from nogil_kit import pool_compare

N, WS = 3_000_000, 8
# 关键坑位已由库内处理：3.14 起 Linux 默认 start_method=forkserver，worker 会重新导入
# __main__；Jupyter 单元格函数挂在不可再导入的 __main__ 上 → forkserver/spawn 直接失败。
# 解法：显式用 fork 上下文（fork 复制内存，无需再导入）；或把 worker 函数放进真实 .py 模块。
cmp = pool_compare(N, WS)
print(f"start_method: {cmp['start_method_default']}（默认）| 进程池使用: {cmp['pool_method']}\n")
print(f"ThreadPool({WS}):   {cmp['thread_seconds']:.3f}s  （无 spawn/pickle 开销）")
print(f"ProcessPool({WS}):   {cmp['process_seconds']:.3f}s  （含 worker 启动 + 参数 pickle，"
      f"约 {cmp['process_over_thread']:.1f}x 线程池耗时）")

## 坑点速查

| 场景 | 用线程（nogil） | 用进程 |
|---|---|---|
| 短任务（<1s） | ✅ 首选 | ❌ spawn 开销占比大 |
| 长任务 / 共享大数据 | ✅ 零拷贝 | ⚠️ pickle 成本，用 `shared_memory` |
| 隔离需求（防崩溃/泄漏） | ❌ | ✅ |
| kernel 里跑且 GIL 已被拉起 | ❌ 退化为 1x | ✅ worker 子进程干净 |

其他坑：
- **3.14 forkserver 默认值打破 notebook 进程池**：单元格函数挂在不可再导入的 `__main__`，forkserver/spawn 报 `FileNotFoundError`/`AttributeError`——用 `mp_context=get_context("fork")` 或把 worker 写进 .py 模块（`nogil_kit` 已内置处理）
- **worker 也会被污染**：worker 子进程若 import 未适配模块，同样 GIL 化——用诊断脚本在 worker 环境复检
- **超订**：`nogil 线程 × 进程池 worker` 会叠加竞争，总并发 ≤ 核数